# Solutions to Exercises on Inference in Bayesian Networks

1. Given the factorised distribution

$P(C,R,D,A,S,F,T) = P(R)P(C)P(D)P(A\vert C)P(F\vert A)P(T\vert F,D) P(S\vert F,T)$

and the following univariate distributions:


| $P(R)$ | $P(\lnot R)$ |
|:------:|:------------:|
| 0.3    | 0.7          |

<br/>

| $P(C)$ | $P(\lnot C)$ |
|:------:|:------------:|
| 0.1    | 0.9          |

<br/>

| $P(D)$ | $P(\lnot D)$ |
|:------:|:------------:|
| 0.5    | 0.5          |

<br/>

|           | $P(A\vert C)$ | $P(\lnot A\vert C)$ |
|----------:|:-------------:|:-------------------:|
| $C$       | 0.01          |  0.99               |
| $\lnot C$ | 0.0           |  1.00               |

<br/>

|           | $P(F\vert A)$ | $P(\lnot F\vert A)$ |
|----------:|:-------------:|:-------------------:|
| $A$       | 0.30          |  0.70               |
| $\lnot A$ | 0.001         |  0.999              |

<br/>

|                  | $P(T\vert F,D)$ | $P(\lnot T\vert F,D)$ |
|-----------------:|:---------------:|:---------------------:|
| $D,F$            | 0.99            |  0.01                 |
| $D,\lnot F$      | 0.60            |  0.40                 |
| $\lnot D,F$      | 0.70            |  0.30                 |
| $\lnot D,\lnot F$| 0.01            |  0.99                |

<br/>

|                  | $P(S\vert T,F)$ | $P(\lnot S\vert T,F)$ |
|-----------------:|:---------------:|:-------------------:|
| $T,F$            | 0.99            |  0.01                 |
| $T,\lnot F$      | 0.30            |  0.70                 |
| $\lnot T,F$      | 0.99            |  0.01                 |
| $\lnot T,\lnot F$| 0.001           |  0.999                |

Compute the following quantities:


In [1]:
import numpy as np
PR = np.array([0.7,0.3])
PC = np.array([0.9,0.1])
PD = np.array([0.5,0.5])
PA_C = np.array([[1.0, 0.99],[0.0,0.01]])
PA = PA_C @ PC
PF_A = np.array([[0.999, 0.7],[0.001,0.3]])
PT_FD = np.array([[[0.99,0.40],[0.30,0.01]],[[0.01,0.60],[0.70,0.99]]])
PF = PF_A @ PA
PT_F = PT_FD @ PD
PT = PT_F @ PF

a) $P(S)$

$P(S)=\sum_{TF}P(S,T,F)=P(S\vert(T,F)P(T)P(F)$

In [2]:
PS_TF = np.array([[[0.999,0.01],[0.70,0.01]],[[0.001,0.99],[0.30,0.99]]])
# Check
print(PS_TF[1][1][0])

PS_T = PS_TF@PF
print(PS_T)

PS = PS_T @ PT
print(PS)


0.3
[[0.99771529 0.69910369]
 [0.00228471 0.30089631]]
[0.90642929 0.09357071]


b) $P(S,C)$


$P(S,C) = P(S\vert C)P(C) = \sum_{FDA} P(S\vert T,F) P(T\vert F,D) P(F\vert A) P(A\vert C) P(D) P(C)$

This is best handled in stages. We will also use some of the things we have already calculated to reformulate this as:

$P(S,C) = \sum({TFA}P(S\vert T,F) P(T\vert F) T(F) P(A\vert C) P(C)$

where $P(S\vert C) = \sum({TFA}P(S\vert T,F) P(T\vert F) T(F) P(A\vert C)$

In [3]:
PT_A = PT_F @ PF_A
print(PT_A)

PS_T = PS_TF @ PF
print(PS_T)

PS_A = PS_T @ PT_A
print(PS_A)

PS_C = PS_A @ PA_C
print(PS_C)

PSC = PS_C*PC
print(PSC)


[[0.69446 0.533  ]
 [0.30554 0.467  ]]
[[0.99771529 0.69910369]
 [0.00228471 0.30089631]]
[[0.9064775  0.85826367]
 [0.0935225  0.14173633]]
[[0.9064775  0.90599536]
 [0.0935225  0.09400464]]
[[0.81582975 0.09059954]
 [0.08417025 0.00940046]]


c) $P(A\vert S)$

For this we will need to use Bayes' rule:

$P(A\vert S) = P(S\vert A)P(A)/P(S)$.

In [4]:
PSA = PS_A * PA
print(PS_A)
print(PA)
print(PSA/PA)

PA_S = PSA.T/PS

print(PA_S)


[[0.9064775  0.85826367]
 [0.0935225  0.14173633]]
[0.999 0.001]
[[0.9064775  0.85826367]
 [0.0935225  0.14173633]]
[[9.99053138e-01 9.98485249e-01]
 [9.46862248e-04 1.51475097e-03]]


d) $P(A\vert T)$

Again, we use Bayes rule: $P(A\vert T) = P(T\vert A)P(A)/P(T)$:

In [5]:
PTA = PT_A * PA
PA_T = PTA.T/PT
print(PA_T)

[[9.99232319e-01 9.98472366e-01]
 [7.67681292e-04 1.52763418e-03]]


2, Write a program to use direct sampling to compute the value of $\pi$.

To help you get started, use the following facts:

* A circle has area $\pi r^2$, where $r$ is the radius of the circle.
* The minimal square that fully contains a circle of radio $r$ has area $4r^2$.
* The function `numpy.random.uniform` samples from a uniform distribution.
* The function `numpy.where` allows a numpy array to be filtered for a condition.

In [6]:
import numpy as np

# Sample points from within a square of side length 2
npts = 100000000
x = np.random.uniform(-1., 1., size=(npts,2))

# How many of them are in the circle
in_circle = x[np.where(x[:,0]*x[:,0]+x[:,1]*x[:,1] <= 1)]

# Estimate value of Pi
est_pi = 4*in_circle.shape[0]/npts
print(est_pi)

3.14170104


3. Given the factorised distribution

$P(C,R,D,A,S,F,T) = P(R)P(C)P(D)P(A\vert C)P(F\vert A)P(T\vert F,D) P(S\vert F,T)$

and the univariate distributions given in the code cell below, compute the following quantities using rejection sampling. Verify your answers against the exact solutions computed in Question 1.

In [7]:
import numpy as np
PR = np.array([0.7,0.3])
PC = np.array([0.9,0.1])
PD = np.array([0.5,0.5])
PA_C = np.array([[1.0, 0.99],[0.0,0.01]])
PF_A = np.array([[0.999, 0.7],[0.001,0.3]])
PT_FD = np.array([[[0.99,0.40],[0.30,0.01]],[[0.01,0.60],[0.70,0.99]]])
PS_TF = np.array([[[0.999,0.01],[0.70,0.01]],[[0.001,0.99],[0.30,0.99]]])


In [8]:
# First, generate the trace
N = 100000
Trace = np.zeros([N,7])
for i in range(N):
    R = np.random.binomial(1, PR[1])
    C = np.random.binomial(1, PC[1])
    D = np.random.binomial(1, PD[1])
    A = np.random.binomial(1, PA_C[1,C])
    F = np.random.binomial(1, PF_A[1,A])
    T = np.random.binomial(1, PT_FD[1,F,D])
    S = np.random.binomial(1, PS_TF[1,T,F])
    Trace[i:] = np.array([R,C,D,A,F,T,S])
    

a) $P(S)$

In [9]:
# Exact calculation
PA = PA_C @ PC
PF = PF_A @ PA
PT_F = PT_FD @ PD
PT = PT_F @ PF
PS_T = PS_TF@PF
PS = PS_T @ PT

print(f"P(S) = {Trace[:,6].sum()/N} (expect {PS[1]})")


P(S) = 0.093 (expect 0.09357071278723453)


b) $P(S,C)$

In [10]:
# Exact calculation
PT_A = PT_F @ PF_A
PS_T = PS_TF @ PF
PS_A = PS_T @ PT_A
PS_C = PS_A @ PA_C
PSC = PS_C*PC

# P(S=1,C=1)
CEq1 = Trace[np.where(Trace[:,1]==1)]
SEq1CEq1 = CEq1[np.where(CEq1[:,6]==1)]
PSEq1CEq1 = SEq1CEq1.shape[0]/N
print(f"P(S=1,C=1) = {PSEq1CEq1} (expect {PSC[1,1]})")

P(S=1,C=1) = 0.00992 (expect 0.009400463724620538)


c) $P(A\vert S)$

In [11]:
PSA = PS_A * PA
PA_S = PSA.T/PS
#P(A=0|S=0)
SEq0 = Trace[np.where(Trace[:,6]==0)]
AEq0SEq0 = SEq0[np.where(SEq0[:,3]==0)]
PAEq0GivenSEq0 = AEq0SEq0.shape[0]/SEq0.shape[0]
print(f"P(A=0|S=0) = {PAEq0GivenSEq0} (expect {PA_S[1,1]})")

P(A=0|S=0) = 0.9996030871003307 (expect 0.0015147509675948146)


d) $P(A\vert T)$

4. Given the factorised distribution

$P(C,R,D,A,S,F,T) = P(R)P(C)P(D)P(A\vert C)P(F\vert A)P(T\vert F,D) P(S\vert F,T)$

and the univariate distributions given in the code cell below, compute the following quantities using likelihood reweighting. Verify your answers against the exact solutions computed in Question 1, and the direct sampling solutions computed in Question 3

In [12]:
import numpy as np
PR = np.array([0.7,0.3])
PC = np.array([0.9,0.1])
PD = np.array([0.5,0.5])
PA_C = np.array([[1.0, 0.99],[0.0,0.01]])
PF_A = np.array([[0.999, 0.7],[0.001,0.3]])
PT_FD = np.array([[[0.99,0.40],[0.30,0.01]],[[0.01,0.60],[0.70,0.99]]])
PS_TF = np.array([[[0.999,0.01],[0.70,0.01]],[[0.001,0.99],[0.30,0.99]]])


a) $P(S\vert C=1, D=0)$

In [13]:
N = 10000
Trace = np.zeros([N,8])
for i in range (N):
    w = 1.0
    R = np.random.binomial(1,PR[1])
    C = 1
    w *= PC[C]
    D = 0
    w *= PD[D]
    A = np.random.binomial(1,PA_C[1,C])
    F = np.random.binomial(1,PF_A[1,A])
    T = np.random.binomial(1,PT_FD[1,F,D])
    S = np.random.binomial(1,PS_TF[1,T,F])
    Trace[i,:] = [R,C,D,A,F,T,S,w]

# P(S|C=1,D=0)
print(f"P(S|C=1,D=0) = {Trace[np.where(Trace[:,6]==1)].shape[0]/N}")

P(S|C=1,D=0) = 0.0084


b) $P(S,C\vert A=1, R=0)$

In [14]:
N = 50000
Trace = np.zeros([N,8])
for i in range (N):
    w = 1.0
    R = 0 
    w *= PR[R]
    C = np.random.binomial(1,PC[1])
    D = np.random.binomial(1,PD[1])
    A = 1
    w *= PA_C[A,C]
    F = np.random.binomial(1,PF_A[1,A])
    T = np.random.binomial(1,PT_FD[1,F,D])
    S = np.random.binomial(1,PS_TF[1,T,F])
    Trace[i,:] = [R,C,D,A,F,T,S,w]

# P(S=1,C=1 | A=1, R=0)
CEq1 = Trace[np.where(Trace[:,1]==1)]
SEq1CEq1 = CEq1[np.where(CEq1[:,6]==1)]
PSEq1CEq1 = SEq1CEq1[:,7].sum() / Trace[:,7].sum()
print(PSEq1CEq1)


0.3542902332937922


c) $P(A\vert S, D=1, R=0)$

In [15]:
N = 50000
Trace = np.zeros([N,8])
for i in range (N):
    w = 1.0
    R = 0 
    w *= PR[R]
    C = np.random.binomial(1,PC[1])
    D = 1
    w *= PD[D]
    A = np.random.binomial(1,PA_C[1,C])
    F = np.random.binomial(1,PF_A[1,A])
    T = np.random.binomial(1,PT_FD[1,F,D])
    S = np.random.binomial(1,PS_TF[1,T,F])
    Trace[i,:] = [R,C,D,A,F,T,S,w]

# P(A=1 | S=1, D=1, R=0)
SEq1 = Trace[np.where(Trace[:,6]==1)]
AEq1SEq1 = SEq1[np.where(SEq1[:,3]==1)]
PAEq1SEq1 = AEq1SEq1[:,7].sum() / SEq1[:,7].sum()
print(PAEq1SEq1)


0.0015540015540015536


d) $P(A\vert T, C=0, D=1)$

In [16]:
N = 50000
Trace = np.zeros([N,8])
for i in range (N):
    w = 1.0
    R = np.random.binomial(1,PR[1])
    C = 0
    w *= PC[C]
    D = 1
    w *= PD[D]
    A = np.random.binomial(1,PA_C[1,C])
    F = np.random.binomial(1,PF_A[1,A])
    T = np.random.binomial(1,PT_FD[1,F,D])
    S = np.random.binomial(1,PS_TF[1,T,F])
    Trace[i,:] = [R,C,D,A,F,T,S,w]

# P(A=0 | T=1, C=0, D=1)
TEq1 = Trace[np.where(Trace[:,5]==1)]
AEq0TEq1 = TEq1[np.where(TEq1[:,3]==0)]
PAEq0TEq1 = AEq0TEq1[:,7].sum() / TEq1[:,7].sum()
print(PAEq0TEq1)


1.0


In [17]:
PTA = PT_A * PA
PA_T = (PTA.T)/PT

TEq1 = Trace[np.where(Trace[:,5]==1)]
AEq0AEq1 = TEq1[np.where(TEq1[:,3]==0)]
PAEq0AEq1 = AEq0AEq1.shape[0]/TEq1.shape[0]
print(f"P(A=0|T=1) = {PAEq0AEq1} (expect {PA_T[0,1]})")
print(PA_T[1,1])

P(A=0|T=1) = 1.0 (expect 0.9984723658172912)
0.0015276341827088428
